In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/XRF_databases/bank_notes/plsda/bank_notes.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '2.74':'22.71']

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '2.74':'22.71'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '2.74':'22.71'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-09 05:22:06,029 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-09 05:22:06,169 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [2]:
from modeling import mlp_optimized

mlp_model = mlp_optimized(Xcalclass_prep, ycalclass, Xpredclass_prep, ypredclass, 
                          aim='classification', 
                          hidden_layer_sizes=(64,32),
                          activation='tanh',
                          learning_rate='adaptive',
                          max_iter=10,
                          random_state=1)
mlp_model[0]

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


,Model,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred
0,MLP,1.0,1.0,1.0,"[[175, 0], [0, 109]]",1.0,1.0,1.0,"[[76, 0], [0, 47]]"


## Definição das Zonas Espectrais

In [3]:
spectral_cuts = [
('Ar ka + Ag L', 2.76, 3.47),
('Ca ka', 3.5, 3.91),
('Ca kb', 3.93, 4.24),
('Ti ka', 4.26, 4.72),
('Ti kb', 4.75, 5.13),
('background1', 5.16, 6.12),
('Fe ka', 6.15, 6.76),
('Fe kb', 6.79, 7.32),
('background2', 7.35, 7.78),
('Cu ka', 7.81, 8.29),
('Zn ka', 8.29, 8.80),
('Cu kb', 8.80, 9.26),
('Zn kb', 9.26, 10.00),
('background3', 10.00, 21.46),
('Ag ka scattering', 21.49, 22.71)
]

## SHAP

In [5]:
shap_global_importance = pd.read_csv('shap_bank_notes.csv', sep=';') # loading previously saved shap_unique_df
# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
shap_unique_df = shap_unique_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df

,energy,Mean_Abs_SHAP,Zone
0,3.73,0.041364,Ca ka
1,4.54,0.028252,Ti ka
2,6.38,0.013990,Fe ka
3,8.09,0.007152,Cu ka
4,4.88,0.003056,Ti kb
5,7.04,0.002463,Fe kb
6,22.28,0.002394,Ag ka scattering
7,4.06,0.000760,Ca kb
8,19.40,0.000475,background3
9,9.33,0.000432,Zn kb


## Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [6]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = exp.aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

Zona 'Ar ka + Ag L': VE = 5.85%, dim = 29 variáveis
Zona 'Ca ka': VE = 98.41%, dim = 17 variáveis
Zona 'Ca kb': VE = 90.24%, dim = 13 variáveis
Zona 'Ti ka': VE = 96.36%, dim = 19 variáveis
Zona 'Ti kb': VE = 80.62%, dim = 16 variáveis
Zona 'background1': VE = 12.07%, dim = 39 variáveis
Zona 'Fe ka': VE = 99.63%, dim = 25 variáveis
Zona 'Fe kb': VE = 96.66%, dim = 22 variáveis
Zona 'background2': VE = 8.40%, dim = 18 variáveis
Zona 'Cu ka': VE = 97.03%, dim = 20 variáveis
Zona 'Zn ka': VE = 20.69%, dim = 21 variáveis
Zona 'Cu kb': VE = 63.49%, dim = 19 variáveis
Zona 'Zn kb': VE = 5.72%, dim = 30 variáveis
Zona 'background3': VE = 2.51%, dim = 451 variáveis
Zona 'Ag ka scattering': VE = 27.73%, dim = 49 variáveis

Scores DataFrame shape: (284, 15)


In [7]:
y_pred = mlp_model[4]['MLP'].values # using the continuous predictions from SVM, extracting as 1D array
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_scores_df,
    y_predicted_numeric=y_pred
)

In [8]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

y_pred = pd.Series(mlp_model[4]['MLP'].values) # using the continuous predictions from SVM, extracting as 1D array

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graphv3_pca(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Fe ka > -23.71,Fe ka > -23.71,Fe ka > -23.71,Fe ka > -23.71
1,Fe ka > -24.15,Fe ka > -24.15,Fe ka > -24.15,Fe ka > -24.15
2,Fe ka > -24.62,Fe ka > -24.62,Fe ka > -24.62,Fe ka > -24.62
3,Ca ka > -26.11,Ca ka > -26.11,Ca ka > -26.11,Ca ka > -21.19
4,Ca ka > -21.19,Ca ka > -21.19,Ca ka > -21.19,Ca ka > -26.11
...,...,...,...,...
87,Zn kb <= 1.07,Zn kb <= 0.29,background2 > -0.37,Fe kb <= -7.41
88,Zn kb <= -0.29,Fe kb <= -7.41,Zn kb <= 0.29,Zn kb > -1.05
89,Zn kb <= 0.29,Zn kb <= 1.07,Zn kb > 0.29,background2 <= 0.27
90,Class_A,Class_A,Class_A,Class_A


In [9]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_cov = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_cov = lrc_summed_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df_cov.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
#lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Fe ka > -23.71,15.988192,Fe ka,-23.71,>
1,Ca ka > -26.11,6.489697,Ca ka,-26.11,>
2,Fe kb > -7.02,5.279888,Fe kb,-7.02,>
3,Ti ka <= 19.85,3.638187,Ti ka,19.85,<=
4,Ca kb > -6.07,2.314792,Ca kb,-6.07,>
5,Cu ka > -10.68,1.694605,Cu ka,-10.68,>
6,Ti kb <= 6.95,0.731985,Ti kb,6.95,<=
7,Cu kb > -2.30,0.177316,Cu kb,-2.30,>
8,Ag ka scattering <= 3.49,0.128208,Ag ka scattering,3.49,<=
9,background3 <= -0.30,0.045245,background3,-0.30,<=


In [10]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zones_original = exp.aggregate_spectral_zones_pca(spectral_zones_original) # apca aggregation

# Aplicar o mapeamento
lrc_summed_df_cov_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_cov,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_cov_natural

Zona 'Ar ka + Ag L': VE = 9.76%, dim = 29 variáveis
Zona 'Ca ka': VE = 99.20%, dim = 17 variáveis
Zona 'Ca kb': VE = 93.13%, dim = 13 variáveis
Zona 'Ti ka': VE = 98.39%, dim = 19 variáveis
Zona 'Ti kb': VE = 85.13%, dim = 16 variáveis
Zona 'background1': VE = 12.27%, dim = 39 variáveis
Zona 'Fe ka': VE = 99.86%, dim = 25 variáveis
Zona 'Fe kb': VE = 97.91%, dim = 22 variáveis
Zona 'background2': VE = 8.43%, dim = 18 variáveis
Zona 'Cu ka': VE = 98.29%, dim = 20 variáveis
Zona 'Zn ka': VE = 24.14%, dim = 21 variáveis
Zona 'Cu kb': VE = 66.75%, dim = 19 variáveis
Zona 'Zn kb': VE = 7.95%, dim = 30 variáveis
Zona 'background3': VE = 2.98%, dim = 451 variáveis
Zona 'Ag ka scattering': VE = 38.41%, dim = 49 variáveis


,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Fe ka > -23.71,15.988192,Fe ka,-23.71,>,-228.336686,46.0,0.003068,Fe ka > -228.336686
1,Fe ka > -24.15,11.434537,Fe ka,-24.15,>,-232.833909,112.0,0.002150,Fe ka > -232.833909
2,Fe ka > -24.62,8.060821,Fe ka,-24.62,>,-240.227437,43.0,0.003827,Fe ka > -240.227437
3,Ca ka > -26.11,6.489697,Ca ka,-26.11,>,-287.239935,110.0,0.000994,Ca ka > -287.239935
4,Ca ka > -21.19,6.180080,Ca ka,-21.19,>,-238.439137,152.0,0.149595,Ca ka > -238.439137
...,...,...,...,...,...,...,...,...,...
87,Fe kb <= -7.41,0.008405,Fe kb,-7.41,<=,-32.289144,4.0,0.001335,Fe kb <= -32.289144
88,background2 <= 0.27,0.008022,background2,0.27,<=,0.811175,47.0,0.000881,background2 <= 0.811175
89,Zn kb <= 1.07,0.007872,Zn kb,1.07,<=,-2.546607,108.0,0.003444,Zn kb <= -2.546607
90,Class_A,0.000000,None,None,None,NaN,NaN,NaN,None


## Reconstrução de Thresholds Multivariados

Agora vamos pegar os thresholds dos predicados mais importantes e reconstruí-los como espectros completos.

In [11]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_cov_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_cov_natural.iloc[n]['Threshold_Natural'])  # Corrigido: usa n em vez de 7

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[0]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_cov_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Fe ka':
  - Dimensão: 25 variáveis espectrais
  - Range de energias: 6.15 - 6.76
  - Variância explicada pela PC1: 99.86%


In [12]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=mlp_model[3],
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='median', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='classification',
        metric='probability_shift', 
        verbose=True
    )

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graphv3_pca(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )

    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 90 | Descartados: 30
PERTURBATION IMPORTANCE PARA PREDICADOS
Tipo de tarefa (aim): classification
Modo de perturbação: median
Fonte das estatísticas: full
Métrica: probability_shift
Total de fold

,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Ca ka > -6.65,Ca ka > -6.65,Ca ka > -6.65,Ca ka > -6.65
1,Ti ka <= -4.69,Ti ka <= -4.69,Ti ka <= -4.69,Ti ka <= -4.69
2,Ti ka <= 9.34,Ca ka > -21.19,Ti ka <= 9.34,Ti ka <= 9.34
3,Ca ka > -26.11,Ti ka <= 9.34,Ca ka > -21.19,Ca ka > -26.11
4,Ca ka > -21.19,Ca ka > -26.11,Ca ka > -26.11,Ca ka > -21.19
...,...,...,...,...
87,Cu kb > -2.30,Cu kb > -2.30,Zn kb > 0.29,background2 > -0.37
88,background2 <= 0.27,background2 <= 0.27,Cu kb > -2.30,Cu kb > -2.30
89,background2 <= -0.37,background2 > -0.92,background2 <= -0.37,background2 <= -0.37
90,Class_A,Class_A,Class_A,Class_A


In [13]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,Ca ka > -6.65,0.060982
1,Ti ka <= -4.69,0.060808
2,Ti ka <= 9.34,0.042351
3,Ca ka > -21.19,0.040061
4,Ti ka <= 19.85,0.032945
...,...,...
85,Cu kb > -1.61,0.000779
86,Cu kb > -2.88,0.000747
87,background2 <= 0.27,0.000736
88,Cu kb > -2.30,0.000709


In [14]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_pert = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_pert = lrc_summed_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df_pert.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Ca ka > -6.65,10.655246,Ca ka,-6.65,>
1,Ti ka <= -4.69,7.334458,Ti ka,-4.69,<=
2,background3 <= -0.30,2.112321,background3,-0.30,<=
3,Cu ka > -8.43,2.080085,Cu ka,-8.43,>
4,Ag ka scattering <= -0.18,1.706129,Ag ka scattering,-0.18,<=
5,Ti kb <= -2.91,1.337856,Ti kb,-2.91,<=
6,background1 <= -1.03,0.203091,background1,-1.03,<=
7,Fe ka > -23.71,0.201094,Fe ka,-23.71,>
8,Fe kb > -7.02,0.193328,Fe kb,-7.02,>
9,Ca kb <= -6.07,0.182308,Ca kb,-6.07,<=


In [15]:
# Aplicar o mapeamento
lrc_summed_df_pert_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_pert,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_pert_natural

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Ca ka > -6.65,10.655246,Ca ka,-6.65,>,-72.180511,102.0,0.000511,Ca ka > -72.180511
1,Ti ka <= -4.69,7.334458,Ti ka,-4.69,<=,-49.917691,64.0,0.070519,Ti ka <= -49.917691
2,Ti ka <= 9.34,5.280687,Ti ka,9.34,<=,101.580304,11.0,0.037350,Ti ka <= 101.580304
3,Ca ka > -21.19,4.919432,Ca ka,-21.19,>,-238.439137,152.0,0.149595,Ca ka > -238.439137
4,Ca ka > -26.11,3.974427,Ca ka,-26.11,>,-287.239935,110.0,0.000994,Ca ka > -287.239935
...,...,...,...,...,...,...,...,...,...
87,background2 <= 0.27,0.071704,background2,0.27,<=,0.811175,47.0,0.000881,background2 <= 0.811175
88,Cu kb > -2.30,0.041151,Cu kb,-2.30,>,-8.474542,221.0,0.000430,Cu kb > -8.474542
89,background2 <= -0.37,0.019814,background2,-0.37,<=,-1.184782,277.0,0.010577,background2 <= -1.184782
90,Class_A,0.000000,None,None,None,NaN,NaN,NaN,None


In [16]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_pert_natural.iloc[n]['Zone'] # o n indica a linha do dataframe lrc_summed_df_pert_natural que queremos acessar para pegar o nome da zona espectral (coluna 'Zone')
threshold_score = float(lrc_summed_df_pert_natural.iloc[n]['Threshold_Natural']) # o iloc acessa a linha n e a coluna 'Threshold_Natural' para pegar o valor do threshold já mapeado para o espaço natural (não pré-processado)

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
#print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_pert_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Ca ka':
  - Dimensão: 17 variáveis espectrais
  - Variância explicada pela PC1: 99.20%


In [18]:
# Permutation importance baseado em mudança nas probabilidades previstas (predict_proba)
# Medimos a média da diferença absoluta entre as probabilidades originais e as probabilidades
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Probabilidades base (classe 'B' mapeada para 1)
baseline_proba = mlp_model[3].predict_proba(Xcalclass_prep)[:, 1]
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_proba = mlp_model[3].predict_proba(X_perm)[:, 1]
        diffs.append(np.mean(np.abs(baseline_proba - perm_proba)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance_proba': importance_list
})
permutation_df.sort_values(by='Permutation_importance_proba', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance_proba', ascending=False)
permutation_unique_df

,energy,Permutation_importance_proba,Zone
0,6.38,0.018148,Fe ka
1,3.73,0.005747,Ca ka
2,4.49,0.004113,Ti ka
3,8.06,0.002176,Cu ka
4,7.07,0.001709,Fe kb
5,4.88,0.001061,Ti kb
6,4.06,0.001044,Ca kb
7,22.28,0.000854,Ag ka scattering
8,8.98,0.000675,Cu kb
9,12.75,0.000597,background3


In [19]:
import numpy as np

max_len = max(
    len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation' : pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,Shap,Permutation,LRC_perturbation,LRC_covariance
0,Ca ka,Fe ka,Ca ka,Fe ka
1,Ti ka,Ca ka,Ti ka,Ca ka
2,Fe ka,Ti ka,background3,Fe kb
3,Cu ka,Cu ka,Cu ka,Ti ka
4,Ti kb,Fe kb,Ag ka scattering,Ca kb
5,Fe kb,Ti kb,Ti kb,Cu ka
6,Ag ka scattering,Ca kb,background1,Ti kb
7,Ca kb,Ag ka scattering,Fe ka,Cu kb
8,background3,Cu kb,Fe kb,Ag ka scattering
9,Zn kb,background3,Ca kb,background3


In [20]:
import rbo
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from the lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Truncate both lists to the same length (minimum of both)
        min_len = min(len(list_1), len(list_2))
        list_1_trunc = list_1[:min_len]
        list_2_trunc = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trunc, list_2_trunc).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_10992\3097109587.py:16: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Method_1,Method_2,RBO_Score
4,Permutation,LRC_covariance,0.871130
1,Shap,LRC_perturbation,0.832015
0,Shap,Permutation,0.544172
2,Shap,LRC_covariance,0.443549
3,Permutation,LRC_perturbation,0.420052
5,LRC_perturbation,LRC_covariance,0.319429


In [21]:
lrc_summed_df_cov_natural.to_csv('lrc_cov_natural.csv', index=False, sep=';')
lrc_summed_df_pert_natural.to_csv('lrc_pert_natural.csv', index=False, sep=';')